In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os, ast, sqlite3, re

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)

In [2]:
def cleanText(text):
    return re.sub(r'[^A-Za-z0-9]', '', text).strip().lower()

# EXTRAINDO DADOS

In [3]:
conn = sqlite3.connect('files/db.sqlite3')
cursor = conn.cursor()

In [4]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = [row[0].split('_')[-1] for row in cursor.fetchall() if 'Etilometros_' in row[0]]
tables

['company',
 'firmware',
 'sensor',
 'sensorreplacelog',
 'solution',
 'calibration',
 'log',
 'etilometro',
 'suntech',
 'device']

In [5]:
dataframes = {key: pd.read_sql_query(f'SELECT * FROM Etilometros_{key}', conn) for key in tables}

In [6]:
dataframes['device'].head()

,id,sensor_id,series_num,suntech_id,need_update,timestamp,company_id,location,chip,default_settings,deleted,srv_created_at,update_settings,updated_at,software_version
0,MIC6211137820391242,ETL1317714633862637,00010,NaN,1,2024-06-20 13:31:38,07295525000140,,,NaN,0,2025-12-10 19:36:52.519550,NaN,2025-12-10 19:36:52.528153,1.0.0
1,MIC0775298835821553,ETL5152384908461030,00053,NaN,0,2024-07-03 13:24:44,29863420000930,,,NaN,0,2025-12-10 19:36:52.519550,NaN,2025-12-10 19:36:52.528153,1.0.0
2,MIC1009016058992021,ETL8355798954801671,00012,NaN,1,2024-07-05 21:21:45,29863420000930,,,NaN,0,2025-12-10 19:36:52.519550,NaN,2025-12-10 19:36:52.528153,1.0.0
3,MIC4598591955329199,ETL7037009852791407,00015,NaN,1,2024-07-05 22:27:18,44922499000160,,,NaN,0,2025-12-10 19:36:52.519550,NaN,2025-12-10 19:36:52.528153,1.0.0
4,MIC8971895463541794,ETL3617854052947237,00008,NaN,1,2024-07-22 12:29:54,61.156.501/000,,,,0,2025-12-10 19:36:52.519550,,2026-07-10 12:34:18.510173,1.0.0


In [7]:
dataframes['etilometro'].head()

,id,vehicle_type,vehicle_plate,installation_date,need_update,installer,device_id,telemetry_id,camera_service,is_operating,nickname,deleted,srv_created_at,updated_at,is_active
0,0968240db3e34c22adc9835991751edb,0,LUA5E58,2024-07-12 14:31:46,0,Klauss,MIC1009016058992021,17131425000132,,1,,0,2025-12-10 19:36:52.540510,2026-04-24 14:26:55.862032,1
1,a671296be54848728e72e56ab38384b8,1,SR07H79,2024-07-15 14:18:16,need_update,,MIC0420088851272205,17131425000132,,1,,0,2025-12-10 19:36:52.540510,2025-12-10 19:36:52.546771,1
2,e1199bd519e040408f5b614d9ed38d5e,1,SRJ7J95,2024-07-15 15:25:54,1,None,MIC1700087363643403,17131425000132,None,1,,0,2025-12-10 19:36:52.540510,2025-12-10 19:36:52.546771,1
3,567557a92d964b3fa1ece239b82bfc63,0,LUE9A29,2024-07-23 11:40:40,0,Leticia,MIC9935714618981987,17131425000132,None,0,Marcelo,0,2025-12-10 19:36:52.540510,2026-06-02 17:19:25.540978,1
4,a357e49dd44245b2a169355434fe9c83,1,PPQ9017,2026-05-06 14:28:37,0,Matheus Vidal,MIC5282201573701056,17131425000132,None,1,,0,2025-12-10 19:36:52.540510,2026-05-06 14:37:34.337508,1


In [8]:
dataframes['log'].head()

,id,event,created_at,etilometro_id,previous_etilometer,previous_plate,deleted,srv_created_at,updated_at,timestamp,lat,lon
0,49474,$ETBL020000!,2025-06-11 21:58:50,d6ec7e0ce9a345a99847728eb1e02e8e,,,0,2025-12-10 19:36:52.797916,2025-12-10 19:36:52.991666,2025-06-12 00:58:52.335119,NaN,NaN
1,49475,$ETEV02!,2025-06-11 21:58:51,d6ec7e0ce9a345a99847728eb1e02e8e,,,0,2025-12-10 19:36:52.797916,2025-12-10 19:36:52.991666,2025-06-12 00:58:53.460542,NaN,NaN
2,49476,$ETEV04!,2025-06-11 21:58:52,d6ec7e0ce9a345a99847728eb1e02e8e,,,0,2025-12-10 19:36:52.797916,2025-12-10 19:36:52.991666,2025-06-12 00:58:55.928921,NaN,NaN
3,49477,$ETEV04!,2025-06-11 22:47:13,27c1c08778e848a79e6709cf33e5726c,,,0,2025-12-10 19:36:52.797916,2025-12-10 19:36:52.991666,2025-06-12 01:47:14.945906,NaN,NaN
4,49478,$ETBL020000!,2025-06-11 22:52:11,27c1c08778e848a79e6709cf33e5726c,,,0,2025-12-10 19:36:52.797916,2025-12-10 19:36:52.991666,2025-06-12 01:52:13.268628,NaN,NaN


# UNIFICANDO DADOS

In [9]:
df = dataframes['device'].copy()
df = df.drop(columns=['need_update', 'location', 'chip', 'default_settings', 'deleted', 'srv_created_at', 'software_version', 'update_settings'])
df.head()

,id,sensor_id,series_num,suntech_id,timestamp,company_id,updated_at
0,MIC6211137820391242,ETL1317714633862637,00010,NaN,2024-06-20 13:31:38,07295525000140,2025-12-10 19:36:52.528153
1,MIC0775298835821553,ETL5152384908461030,00053,NaN,2024-07-03 13:24:44,29863420000930,2025-12-10 19:36:52.528153
2,MIC1009016058992021,ETL8355798954801671,00012,NaN,2024-07-05 21:21:45,29863420000930,2025-12-10 19:36:52.528153
3,MIC4598591955329199,ETL7037009852791407,00015,NaN,2024-07-05 22:27:18,44922499000160,2025-12-10 19:36:52.528153
4,MIC8971895463541794,ETL3617854052947237,00008,NaN,2024-07-22 12:29:54,61.156.501/000,2026-07-10 12:34:18.510173


In [10]:
df['installed'] = df['id'].isin(dataframes['etilometro'].device_id.unique())
df.head()

,id,sensor_id,series_num,suntech_id,timestamp,company_id,updated_at,installed
0,MIC6211137820391242,ETL1317714633862637,00010,NaN,2024-06-20 13:31:38,07295525000140,2025-12-10 19:36:52.528153,False
1,MIC0775298835821553,ETL5152384908461030,00053,NaN,2024-07-03 13:24:44,29863420000930,2025-12-10 19:36:52.528153,False
2,MIC1009016058992021,ETL8355798954801671,00012,NaN,2024-07-05 21:21:45,29863420000930,2025-12-10 19:36:52.528153,True
3,MIC4598591955329199,ETL7037009852791407,00015,NaN,2024-07-05 22:27:18,44922499000160,2025-12-10 19:36:52.528153,False
4,MIC8971895463541794,ETL3617854052947237,00008,NaN,2024-07-22 12:29:54,61.156.501/000,2026-07-10 12:34:18.510173,False


In [11]:
df['series_num'] = df.series_num.astype(int)
df = df.sort_values(by='series_num').reset_index(drop=True)
df.head()

,id,sensor_id,series_num,suntech_id,timestamp,company_id,updated_at,installed
0,MIC4804590476459611,ETL4983647116889969,0,NaN,2025-01-31 19:46:59,44922499000160,2025-12-10 19:36:52.528153,False
1,MIC5403983868841080,ETL4983647116889969,1,NaN,2025-01-31 19:46:59,44922499000160,2025-12-10 19:36:52.528153,False
2,MIC0573510096741149,ETL3802440385517607,2,NaN,2025-04-02 15:14:22,44922499000160,2025-12-10 19:36:52.528153,False
3,MIC9167233510881833,ETL7959493652371592,3,NaN,2025-06-10 17:17:42,44922499000160,2025-12-10 19:36:52.528153,False
4,MIC8767790427121753,ETL4718937222929439,4,693086173433,2026-01-06 12:09:10.395679,02155735000110,2026-08-05 11:53:25.897924,True


# FORMATANDO CAMPOS DE ID

In [12]:
def getTelemetry(value):
    value = str(value).strip()

    if any(c.isalpha() for c in value):
        return 'mix'
    
    if value.isdigit() and len(value) > 10:
        return 'entrack'

    if value.isdigit() and len(value) > 8:
        return 'suntech'
    
    return 'N/A'


df['telemetry'] = df.suntech_id.apply(getTelemetry)
df['suntech']   = [None if telemetry == 'mix' else str(id) for telemetry, id in zip(df.telemetry, df.suntech_id)]
df

,id,sensor_id,series_num,suntech_id,timestamp,company_id,updated_at,installed,telemetry,suntech
0,MIC4804590476459611,ETL4983647116889969,0,NaN,2025-01-31 19:46:59,44922499000160,2025-12-10 19:36:52.528153,False,mix,NaN
1,MIC5403983868841080,ETL4983647116889969,1,NaN,2025-01-31 19:46:59,44922499000160,2025-12-10 19:36:52.528153,False,mix,NaN
2,MIC0573510096741149,ETL3802440385517607,2,NaN,2025-04-02 15:14:22,44922499000160,2025-12-10 19:36:52.528153,False,mix,NaN
3,MIC9167233510881833,ETL7959493652371592,3,NaN,2025-06-10 17:17:42,44922499000160,2025-12-10 19:36:52.528153,False,mix,NaN
4,MIC8767790427121753,ETL4718937222929439,4,693086173433,2026-01-06 12:09:10.395679,02155735000110,2026-08-05 11:53:25.897924,True,entrack,693086173433
...,...,...,...,...,...,...,...,...,...,...
247,MIC4489677039908981,ETL9263776954081852,333,1700009988,2026-07-23 14:26:46.081185,30119676000160,2026-07-23 17:19:22.317096,True,suntech,1700009988
248,MIC6547796052851309,ETL5264760983911053,334,NaN,2026-07-27 12:05:50.537239,44922499000160,2026-07-27 12:06:27.637265,False,mix,NaN
249,MIC5705743242481141,ETL7824069458441565,335,NaN,2026-07-30 12:49:30.006329,44922499000160,2026-07-30 12:51:55.554436,False,mix,NaN
250,MIC2201831250704405,ETL5444940527071089,336,NaN,2026-07-30 17:20:55.504531,44922499000160,2026-07-30 17:22:11.982030,False,mix,NaN


In [13]:
def getCompany(company):
    companies = dataframes['company'].copy()
    target    = companies.loc[companies['id'] == company]

    if len(target) == 0:
        return None
    
    return target.iloc[0].value


dataframes['company']['id'] = dataframes['company']['id'].apply(cleanText)
df['company'] = df.company_id.apply(cleanText)
del df['company_id']

df['company'] = df.company.apply(getCompany)
df.head()

,id,sensor_id,series_num,suntech_id,timestamp,updated_at,installed,telemetry,suntech,company
0,MIC4804590476459611,ETL4983647116889969,0,NaN,2025-01-31 19:46:59,2025-12-10 19:36:52.528153,False,mix,NaN,sighir
1,MIC5403983868841080,ETL4983647116889969,1,NaN,2025-01-31 19:46:59,2025-12-10 19:36:52.528153,False,mix,NaN,sighir
2,MIC0573510096741149,ETL3802440385517607,2,NaN,2025-04-02 15:14:22,2025-12-10 19:36:52.528153,False,mix,NaN,sighir
3,MIC9167233510881833,ETL7959493652371592,3,NaN,2025-06-10 17:17:42,2025-12-10 19:36:52.528153,False,mix,NaN,sighir
4,MIC8767790427121753,ETL4718937222929439,4,693086173433,2026-01-06 12:09:10.395679,2026-08-05 11:53:25.897924,True,entrack,693086173433,leste


In [14]:
def getPlate(row):
    if not row.installed:
        return None

    etilometers = dataframes['etilometro'].copy()
    target      = etilometers.loc[etilometers['device_id'] == row['id']]

    if len(target) == 0:
        return None
    
    return target.iloc[0].vehicle_plate


df['plate'] = [getPlate(row) for i, row in df.iterrows()]
df = df.loc[~df.plate.isna()]
df = df.copy()
df

,id,sensor_id,series_num,suntech_id,timestamp,updated_at,installed,telemetry,suntech,company,plate
4,MIC8767790427121753,ETL4718937222929439,4,693086173433,2026-01-06 12:09:10.395679,2026-08-05 11:53:25.897924,True,entrack,693086173433,leste,TFE6H05
6,MIC7952031891471590,ETL7952031891471590,6,NaN,2026-03-18 11:51:29.539852,2026-03-18 11:51:29.539832,True,mix,NaN,predileto,RJO6I53
10,MIC1009016058992021,ETL8355798954801671,12,NaN,2024-07-05 21:21:45,2025-12-10 19:36:52.528153,True,mix,NaN,predileto,LUA5E58
13,MIC3982343538817967,ETL3178405820036359,22,NaN,2024-09-02 18:16:31,2025-12-10 19:36:52.528153,True,mix,NaN,manchur,KYA9508
14,MIC1700087363643403,ETL0644031280801291,25,NaN,2024-08-14 17:22:06,2025-12-10 19:36:52.528153,True,mix,NaN,logika,SRJ7J95
...,...,...,...,...,...,...,...,...,...,...,...
240,MIC2824659041515651,ETL7016580521711403,325,1700023897,2026-07-15 18:42:33.256790,2026-07-15 18:49:21.741190,True,suntech,1700023897,faje,LUO3I89
241,MIC3746566632057495,ETL3850432974637703,327,693086018547,2026-07-20 12:51:37.237936,2026-07-27 16:38:04.600950,True,entrack,693086018547,sighir,MALETA_SIGHIR
243,MIC7161855067661432,ETL7161855067661432,329,1700006725,2026-07-20 17:15:26.553492,2026-07-20 17:17:00.633705,True,suntech,1700006725,predileto,SRI3E32
246,MIC2964578494695931,ETL6186117015801237,332,1700023886,2026-07-23 13:23:32.545251,2026-07-23 18:07:35.963291,True,suntech,1700023886,ricker,TUH4A08


In [15]:
df.loc[df.plate.str.startswith('RJS5A13')]

,id,sensor_id,series_num,suntech_id,timestamp,updated_at,installed,telemetry,suntech,company,plate
97,MIC9558193970261911,ETL6474449870241295,163,NaN,2025-04-16 16:58:26,2025-12-10 19:36:52.528153,True,mix,NaN,predileto,RJS5A13


In [16]:
def getInstalationDate(row):
    if not row.installed:
        return None
    
    etilometers = dataframes['etilometro'].copy()
    target      = etilometers.loc[etilometers['device_id'] == row['id']]
    return None if len(target) == 0 else target.iloc[0].installation_date


df.loc[:, 'installation_date'] = [getInstalationDate(row) for i, row in df.iterrows()]
df

,id,sensor_id,series_num,suntech_id,timestamp,updated_at,installed,telemetry,suntech,company,plate,installation_date
4,MIC8767790427121753,ETL4718937222929439,4,693086173433,2026-01-06 12:09:10.395679,2026-08-05 11:53:25.897924,True,entrack,693086173433,leste,TFE6H05,2026-07-21 16:20:50
6,MIC7952031891471590,ETL7952031891471590,6,NaN,2026-03-18 11:51:29.539852,2026-03-18 11:51:29.539832,True,mix,NaN,predileto,RJO6I53,2026-03-18 11:51:38
10,MIC1009016058992021,ETL8355798954801671,12,NaN,2024-07-05 21:21:45,2025-12-10 19:36:52.528153,True,mix,NaN,predileto,LUA5E58,2024-07-12 14:31:46
13,MIC3982343538817967,ETL3178405820036359,22,NaN,2024-09-02 18:16:31,2025-12-10 19:36:52.528153,True,mix,NaN,manchur,KYA9508,2024-09-23 12:54:46
14,MIC1700087363643403,ETL0644031280801291,25,NaN,2024-08-14 17:22:06,2025-12-10 19:36:52.528153,True,mix,NaN,logika,SRJ7J95,2024-07-15 15:25:54
...,...,...,...,...,...,...,...,...,...,...,...,...
240,MIC2824659041515651,ETL7016580521711403,325,1700023897,2026-07-15 18:42:33.256790,2026-07-15 18:49:21.741190,True,suntech,1700023897,faje,LUO3I89,2026-07-25 10:00:37
241,MIC3746566632057495,ETL3850432974637703,327,693086018547,2026-07-20 12:51:37.237936,2026-07-27 16:38:04.600950,True,entrack,693086018547,sighir,MALETA_SIGHIR,2026-06-01 17:22:38
243,MIC7161855067661432,ETL7161855067661432,329,1700006725,2026-07-20 17:15:26.553492,2026-07-20 17:17:00.633705,True,suntech,1700006725,predileto,SRI3E32,2026-07-20 17:10:54
246,MIC2964578494695931,ETL6186117015801237,332,1700023886,2026-07-23 13:23:32.545251,2026-07-23 18:07:35.963291,True,suntech,1700023886,ricker,TUH4A08,2026-08-04 13:06:19


In [17]:
def getDate(date):
    if not date:
        return None
    
    date  = str(date)
    sep   = '-' if len(date.split('-')) > 2 else '/'
    date  = date.strip().split(' ')[0].strip()
    
    for part in date.split(sep):
        if not part.isdigit():
            return None

    first, second, third  = date.split(sep)
    year  = first if int(first) > 1000 else third
    day   = third if int(first) > 1000 else first
    month = second
    
    date = f'{year}-{month}-{day}'
    return pd.to_datetime(date, format='%Y-%m-%d', errors='coerce')


df['installation_date'] = df.installation_date.apply(getDate)
df['timestamp'] = df.timestamp.apply(getDate)
df

,id,sensor_id,series_num,suntech_id,timestamp,updated_at,installed,telemetry,suntech,company,plate,installation_date
4,MIC8767790427121753,ETL4718937222929439,4,693086173433,2026-01-06,2026-08-05 11:53:25.897924,True,entrack,693086173433,leste,TFE6H05,2026-07-21
6,MIC7952031891471590,ETL7952031891471590,6,NaN,2026-03-18,2026-03-18 11:51:29.539832,True,mix,NaN,predileto,RJO6I53,2026-03-18
10,MIC1009016058992021,ETL8355798954801671,12,NaN,2024-07-05,2025-12-10 19:36:52.528153,True,mix,NaN,predileto,LUA5E58,2024-07-12
13,MIC3982343538817967,ETL3178405820036359,22,NaN,2024-09-02,2025-12-10 19:36:52.528153,True,mix,NaN,manchur,KYA9508,2024-09-23
14,MIC1700087363643403,ETL0644031280801291,25,NaN,2024-08-14,2025-12-10 19:36:52.528153,True,mix,NaN,logika,SRJ7J95,2024-07-15
...,...,...,...,...,...,...,...,...,...,...,...,...
240,MIC2824659041515651,ETL7016580521711403,325,1700023897,2026-07-15,2026-07-15 18:49:21.741190,True,suntech,1700023897,faje,LUO3I89,2026-07-25
241,MIC3746566632057495,ETL3850432974637703,327,693086018547,2026-07-20,2026-07-27 16:38:04.600950,True,entrack,693086018547,sighir,MALETA_SIGHIR,2026-06-01
243,MIC7161855067661432,ETL7161855067661432,329,1700006725,2026-07-20,2026-07-20 17:17:00.633705,True,suntech,1700006725,predileto,SRI3E32,2026-07-20
246,MIC2964578494695931,ETL6186117015801237,332,1700023886,2026-07-23,2026-07-23 18:07:35.963291,True,suntech,1700023886,ricker,TUH4A08,2026-08-04


In [18]:
logs = pd.merge(
    dataframes['log'], 
    dataframes['etilometro'], 
    left_on='etilometro_id', 
    right_on='id', 
    how='left'
)

logs = pd.merge(
    logs, 
    df, 
    left_on='device_id', 
    right_on='id', 
    how='left'
)

logs

,id_x,event,created_at,etilometro_id,previous_etilometer,previous_plate,deleted_x,srv_created_at_x,updated_at_x,timestamp_x,lat,lon,id_y,vehicle_type,vehicle_plate,installation_date_x,need_update,installer,device_id,telemetry_id,camera_service,is_operating,nickname,deleted_y,srv_created_at_y,updated_at_y,is_active,id,sensor_id,series_num,suntech_id,timestamp_y,updated_at,installed,telemetry,suntech,company,plate,installation_date_y
0,49474,$ETBL020000!,2025-06-11 21:58:50,d6ec7e0ce9a345a99847728eb1e02e8e,,,0,2025-12-10 19:36:52.797916,2025-12-10 19:36:52.991666,2025-06-12 00:58:52.335119,NaN,NaN,d6ec7e0ce9a345a99847728eb1e02e8e,0.0,FGT9B86,06-09-2025 16:00,need_update,Rodrigo Translem,MIC1868677529293739,44922499000,None,1.0,,0.0,2025-12-10 19:36:52.540510,2025-12-10 19:36:52.546771,1.0,MIC1868677529293739,ETL6691950631911338,180.0,1700006724,2025-05-14,2026-04-30 16:52:51.485397,True,suntech,1700006724,klabin,FGT9B86,2025-09-06
1,49475,$ETEV02!,2025-06-11 21:58:51,d6ec7e0ce9a345a99847728eb1e02e8e,,,0,2025-12-10 19:36:52.797916,2025-12-10 19:36:52.991666,2025-06-12 00:58:53.460542,NaN,NaN,d6ec7e0ce9a345a99847728eb1e02e8e,0.0,FGT9B86,06-09-2025 16:00,need_update,Rodrigo Translem,MIC1868677529293739,44922499000,None,1.0,,0.0,2025-12-10 19:36:52.540510,2025-12-10 19:36:52.546771,1.0,MIC1868677529293739,ETL6691950631911338,180.0,1700006724,2025-05-14,2026-04-30 16:52:51.485397,True,suntech,1700006724,klabin,FGT9B86,2025-09-06
2,49476,$ETEV04!,2025-06-11 21:58:52,d6ec7e0ce9a345a99847728eb1e02e8e,,,0,2025-12-10 19:36:52.797916,2025-12-10 19:36:52.991666,2025-06-12 00:58:55.928921,NaN,NaN,d6ec7e0ce9a345a99847728eb1e02e8e,0.0,FGT9B86,06-09-2025 16:00,need_update,Rodrigo Translem,MIC1868677529293739,44922499000,None,1.0,,0.0,2025-12-10 19:36:52.540510,2025-12-10 19:36:52.546771,1.0,MIC1868677529293739,ETL6691950631911338,180.0,1700006724,2025-05-14,2026-04-30 16:52:51.485397,True,suntech,1700006724,klabin,FGT9B86,2025-09-06
3,49477,$ETEV04!,2025-06-11 22:47:13,27c1c08778e848a79e6709cf33e5726c,,,0,2025-12-10 19:36:52.797916,2025-12-10 19:36:52.991666,2025-06-12 01:47:14.945906,NaN,NaN,27c1c08778e848a79e6709cf33e5726c,0.0,EOJ5I60,2026-05-12 12:39:37,0,Fernando,MIC0990885423811983,44922499000,None,1.0,,0.0,2025-12-10 19:36:52.540510,2026-05-12 12:39:38.376878,1.0,MIC0990885423811983,ETL2533584730825069,174.0,1700006712,2025-05-06,2025-12-10 19:36:52.528153,True,suntech,1700006712,bellavia,EOJ5I60,2026-05-12
4,49478,$ETBL020000!,2025-06-11 22:52:11,27c1c08778e848a79e6709cf33e5726c,,,0,2025-12-10 19:36:52.797916,2025-12-10 19:36:52.991666,2025-06-12 01:52:13.268628,NaN,NaN,27c1c08778e848a79e6709cf33e5726c,0.0,EOJ5I60,2026-05-12 12:39:37,0,Fernando,MIC0990885423811983,44922499000,None,1.0,,0.0,2025-12-10 19:36:52.540510,2026-05-12 12:39:38.376878,1.0,MIC0990885423811983,ETL2533584730825069,174.0,1700006712,2025-05-06,2025-12-10 19:36:52.528153,True,suntech,1700006712,bellavia,EOJ5I60,2026-05-12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
460500,703856,$ETEV05!,2026-08-11 10:54:39,3b611d90bd6b420cbbe766679595c242,NaN,NaN,0,2026-08-11 13:54:41.664769,2026-08-11 13:54:41.664803,2026-08-11 13:54:41.664829,-22.869828,-43.112497,3b611d90bd6b420cbbe766679595c242,0.0,TUO7J31,2026-07-25 10:02:28,1,Paulo Ricardo,MIC0857288188281717,44922499000,None,1.0,,0.0,2026-07-25 10:02:49.152939,2026-07-27 16:57:14.559175,1.0,MIC0857288188281717,ETL0686743878371375,319.0,1700010001,2026-06-24,2026-07-27 16:57:30.643311,True,suntech,1700010001,faje,TUO7J31,2026-07-25
460501,703857,$ETEV16!,2026-08-11 10:54:50,b3c83e5d210444b4965252c9c552a6fa,NaN,NaN,0,2026-08-11 13:54:52.385708,2026-08-11 13:54:52.385732,2026-08-11 13:54:52.385759,-22.772681,-42.933224,b3c83e5d210444b4965252c9c552a6fa,0.0,LRA2597,2026-08-04 15:50:17,1,Paulo Ricardo,MIC9481843474461896,44922499000,None,1.0,,0.0,2026-08-04 15:50:41.098628,2026-08-04 15:50:41.098654,1.0,MIC9481843474461

In [19]:
logs = logs[['id_x', 'event', 'timestamp_x', 'device_id', 'installed', 'telemetry', 'company', 'plate', 'suntech']]
logs.columns = ['id', 'event', 'timestamp', 'device', 'installed', 'telemetry', 'company', 'plate', 'suntech']
logs

,id,event,timestamp,device,installed,telemetry,company,plate,suntech
0,49474,$ETBL020000!,2025-06-12 00:58:52.335119,MIC1868677529293739,True,suntech,klabin,FGT9B86,1700006724
1,49475,$ETEV02!,2025-06-12 00:58:53.460542,MIC1868677529293739,True,suntech,klabin,FGT9B86,1700006724
2,49476,$ETEV04!,2025-06-12 00:58:55.928921,MIC1868677529293739,True,suntech,klabin,FGT9B86,1700006724
3,49477,$ETEV04!,2025-06-12 01:47:14.945906,MIC0990885423811983,True,suntech,bellavia,EOJ5I60,1700006712
4,49478,$ETBL020000!,2025-06-12 01:52:13.268628,MIC0990885423811983,True,suntech,bellavia,EOJ5I60,1700006712
...,...,...,...,...,...,...,...,...,...
460500,703856,$ETEV05!,2026-08-11 13:54:41.664829,MIC0857288188281717,True,suntech,faje,TUO7J31,1700010001
460501,703857,$ETEV16!,2026-08-11 13:54:52.385759,MIC9481843474461896,True,suntech,faje,LRA2597,1700009996
460502,703858,$ETEV01!,2026-08-11 13:54:55.309125,MIC9481843474461896,True,suntech,faje,LRA2597,1700009996
460503,703859,$ETEV16!,2026-08-11 13:55:04.574681,MIC0857288188281717,True,suntech,faje,TUO7J31,1700010001


In [20]:
logs = logs.copy()
logs['raw_event'] = [event.replace(' ', '').replace('$', '').replace('!', '').strip().lower() for event in logs.event]
logs['length']    = [len(event) for event in logs.raw_event]
logs['timestamp'] = pd.to_datetime(logs.timestamp, format='mixed')
logs

,id,event,timestamp,device,installed,telemetry,company,plate,suntech,raw_event,length
0,49474,$ETBL020000!,2025-06-12 00:58:52.335119,MIC1868677529293739,True,suntech,klabin,FGT9B86,1700006724,etbl020000,10
1,49475,$ETEV02!,2025-06-12 00:58:53.460542,MIC1868677529293739,True,suntech,klabin,FGT9B86,1700006724,etev02,6
2,49476,$ETEV04!,2025-06-12 00:58:55.928921,MIC1868677529293739,True,suntech,klabin,FGT9B86,1700006724,etev04,6
3,49477,$ETEV04!,2025-06-12 01:47:14.945906,MIC0990885423811983,True,suntech,bellavia,EOJ5I60,1700006712,etev04,6
4,49478,$ETBL020000!,2025-06-12 01:52:13.268628,MIC0990885423811983,True,suntech,bellavia,EOJ5I60,1700006712,etbl020000,10
...,...,...,...,...,...,...,...,...,...,...,...
460500,703856,$ETEV05!,2026-08-11 13:54:41.664829,MIC0857288188281717,True,suntech,faje,TUO7J31,1700010001,etev05,6
460501,703857,$ETEV16!,2026-08-11 13:54:52.385759,MIC9481843474461896,True,suntech,faje,LRA2597,1700009996,etev16,6
460502,703858,$ETEV01!,2026-08-11 13:54:55.309125,MIC9481843474461896,True,suntech,faje,LRA2597,1700009996,etev01,6
460503,703859,$ETEV16!,2026-08-11 13:55:04.574681,MIC0857288188281717,True,suntech,faje,TUO7J31,1700010001,etev16,6


In [21]:
remove = []

In [22]:
target = logs.loc[logs.length > 20]
remove.append(target)
target

,id,event,timestamp,device,installed,telemetry,company,plate,suntech,raw_event,length
437309,680665,$ETEV09ETL5444940527071089!,2026-07-06 20:07:44.562409,MIC3746566632057495,True,entrack,sighir,MALETA_SIGHIR,693086018547,etev09etl5444940527071089,25
437318,680674,$ETEV36ETL4718937222929439!,2026-07-06 20:10:53.676399,MIC3746566632057495,True,entrack,sighir,MALETA_SIGHIR,693086018547,etev36etl4718937222929439,25
437319,680675,$ETEV09ETL4718937222929439!,2026-07-06 20:10:53.980290,MIC3746566632057495,True,entrack,sighir,MALETA_SIGHIR,693086018547,etev09etl4718937222929439,25
437326,680682,$ETEV36ETL5444940527071089!,2026-07-06 20:14:43.472204,MIC3746566632057495,True,entrack,sighir,MALETA_SIGHIR,693086018547,etev36etl5444940527071089,25
437331,680687,$ETEV09ETL5444940527071089!,2026-07-06 20:14:43.975548,MIC3746566632057495,True,entrack,sighir,MALETA_SIGHIR,693086018547,etev09etl5444940527071089,25
437335,680691,$ETEV36ETL4718937222929439!,2026-07-06 20:17:24.301910,MIC3746566632057495,True,entrack,sighir,MALETA_SIGHIR,693086018547,etev36etl4718937222929439,25


In [23]:
# 1. Calcula a data limite (1 ano e 6 meses atrás a partir do momento atual)
limite_data = pd.Timestamp.now() - pd.DateOffset(years=1, months=6)

# 2. Identifica os logs cujo timestamp é mais antigo que essa data limite
target = logs.loc[logs.timestamp < limite_data].copy()

# 3. Adiciona esses logs antigos na sua lista de remoção
remove.append(target)
target

,id,event,timestamp,device,installed,telemetry,company,plate,suntech,raw_event,length


In [24]:
logs = logs.sort_values(by=['device', 'raw_event', 'timestamp'])
logs['diff_tempo'] = logs.groupby(['device', 'raw_event'])['timestamp'].diff()
target = logs.loc[logs['diff_tempo'] <= pd.Timedelta(minutes=2)].copy()

remove.append(target)
target

,id,event,timestamp,device,installed,telemetry,company,plate,suntech,raw_event,length,diff_tempo
411918,655273,$ETEV01!,2026-06-09 13:32:35.342531,MIC0014246467341799,True,suntech,detector,SYM4F89,1700023877,etev01,6,0 days 00:01:52.965360
415956,659312,$ETEV01!,2026-06-12 18:06:47.388396,MIC0014246467341799,True,suntech,detector,SYM4F89,1700023877,etev01,6,0 days 00:01:58.234527
421921,665277,$ETEV01!,2026-06-18 19:10:04.880923,MIC0014246467341799,True,suntech,detector,SYM4F89,1700023877,etev01,6,0 days 00:00:54.150476
424385,667741,$ETEV01!,2026-06-22 19:01:40.193527,MIC0014246467341799,True,suntech,detector,SYM4F89,1700023877,etev01,6,0 days 00:00:49.972573
426169,669525,$ETEV01!,2026-06-24 12:01:07.732669,MIC0014246467341799,True,suntech,detector,SYM4F89,1700023877,etev01,6,0 days 00:01:42.099919
...,...,...,...,...,...,...,...,...,...,...,...,...
445464,688820,$ETEV17!,2026-07-15 17:46:27.580066,MIC9935714618981987,True,mix,predileto,LUE9A29,NaN,etev17,6,0 days 00:00:00.000115
448011,691367,$ETEV17!,2026-07-18 11:29:18.037569,MIC9935714618981987,True,mix,predileto,LUE9A29,NaN,etev17,6,0 days 00:00:00.000043
448012,691368,$ETEV17!,2026-07-18 11:29:18.037622,MIC9935714618981987,True,mix,predileto,LUE9A29,NaN,etev17,6,0 days 00:00:00.000053
451304,694660,$ETEV17!,2026-07-22 14:09:54.666729,MIC9935714618981987,True,mix,predileto,LUE9A29,NaN,etev17,6,0 days 00:00:00.000044


In [25]:
remove = pd.concat(remove)
remove = remove.drop_duplicates(subset=['id'])
remove

,id,event,timestamp,device,installed,telemetry,company,plate,suntech,raw_event,length,diff_tempo
437309,680665,$ETEV09ETL5444940527071089!,2026-07-06 20:07:44.562409,MIC3746566632057495,True,entrack,sighir,MALETA_SIGHIR,693086018547,etev09etl5444940527071089,25,NaT
437318,680674,$ETEV36ETL4718937222929439!,2026-07-06 20:10:53.676399,MIC3746566632057495,True,entrack,sighir,MALETA_SIGHIR,693086018547,etev36etl4718937222929439,25,NaT
437319,680675,$ETEV09ETL4718937222929439!,2026-07-06 20:10:53.980290,MIC3746566632057495,True,entrack,sighir,MALETA_SIGHIR,693086018547,etev09etl4718937222929439,25,NaT
437326,680682,$ETEV36ETL5444940527071089!,2026-07-06 20:14:43.472204,MIC3746566632057495,True,entrack,sighir,MALETA_SIGHIR,693086018547,etev36etl5444940527071089,25,NaT
437331,680687,$ETEV09ETL5444940527071089!,2026-07-06 20:14:43.975548,MIC3746566632057495,True,entrack,sighir,MALETA_SIGHIR,693086018547,etev09etl5444940527071089,25,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...
445464,688820,$ETEV17!,2026-07-15 17:46:27.580066,MIC9935714618981987,True,mix,predileto,LUE9A29,NaN,etev17,6,0 days 00:00:00.000115
448011,691367,$ETEV17!,2026-07-18 11:29:18.037569,MIC9935714618981987,True,mix,predileto,LUE9A29,NaN,etev17,6,0 days 00:00:00.000043
448012,691368,$ETEV17!,2026-07-18 11:29:18.037622,MIC9935714618981987,True,mix,predileto,LUE9A29,NaN,etev17,6,0 days 00:00:00.000053
451304,694660,$ETEV17!,2026-07-22 14:09:54.666729,MIC9935714618981987,True,mix,predileto,LUE9A29,NaN,etev17,6,0 days 00:00:00.000044


In [26]:
remove.to_csv('output/remove.csv', index=None)

# VEÍCULOS INATIVOS

In [27]:
cutoff = pd.Timestamp.now() - pd.DateOffset(months=2)

lastLog = logs.sort_values('timestamp').drop_duplicates(subset='plate', keep='last')[['plate', 'company', 'timestamp']]
lastLog.columns = ['plate', 'company', 'last_log']
lastLog['last_log'] = pd.to_datetime(lastLog['last_log']).dt.strftime('%d/%m/%Y')

inactive = lastLog.loc[pd.to_datetime(lastLog.last_log, dayfirst=True) < cutoff].sort_values('last_log')
print(f'{len(inactive)} veículos inativos (sem logs desde {cutoff.strftime("%d/%m/%Y")})')
inactive

10 veículos inativos (sem logs desde 13/06/2026)


,plate,company,last_log
37724,RJA7A37,felka,03/10/2025
47296,EMP009,predileto,03/11/2025
141197,JAO1G85,ricker,11/02/2026
2226,FGT9B86,klabin,15/06/2025
42833,MOV1T23,felka,17/10/2025
4104,SPX4A02,bellavia,19/06/2025
129064,RJV1A55,predileto,21/01/2026
5476,EOJ5I60,bellavia,23/06/2025
351241,RKG9C05,predileto,28/05/2026
46546,SEQ3J35,klabin,31/10/2025
